# Run RFI processing on selected products

In [ ]:
%pip install aiohttp requests
%load_ext autoreload
%autoreload 2
    
from shapely.geometry import Polygon, shape, mapping
from rasterio.windows import Window
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pystac_client import Client
from folium import GeoJson
import geopandas as gpd
from io import BytesIO
import rasterio as rio
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import requests
import pathlib
import fsspec
import folium
import fiona
import os
import json
from datetime import datetime, timedelta
import time
import yaml
from pathlib import Path
import branca.colormap as cm
import antimeridian

from processing import pol_mean, prescreen_pulses, range_power, range_spectrogram, read_item, read_annot
from plotting import fetch_quicklook, plot_product, plot_result

In [ ]:
# Load the config.yaml to access the paths
cfg_file = Path("../config.yaml").resolve()
cfg = yaml.safe_load(cfg_file.read_text())

# Set the name of the path in the config.yaml
token_path = (cfg_file.parent / cfg["paths"]["token_file"]).resolve()
out_path = (cfg_file.parent / cfg["paths"]["output"]).resolve()
data_path = (cfg_file.parent / cfg["paths"]["data"]).resolve()

rfi_prod_path = f"{out_path}/products_rfi.json"
gdf_path = f"{out_path}/gdf.parquet"

print(f"Token Path: {token_path}")
print(f"Output Path: {out_path}")
print(f"List of RFI products Path: {rfi_prod_path}")
print(f"GeoDataframe of RFI products Path: {gdf_path}")

In [ ]:
# Load the list of products affected by RFI
with open(rfi_prod_path) as f:
    products_rfi = json.load(f)
    
# Load the GeoDataframe
gdf = gpd.read_parquet(gdf_path)

gdf.head()
print(len(gdf))
print(len(products_rfi))

In [ ]:
catalog_url = 'https://catalog.maap.eo.esa.int/catalogue/'
catalog = Client.open(catalog_url)

# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

if pathlib.Path(token_path).exists():
  with open(token_path, "rt") as f:
    OFFLINE_TOKEN = f.read().strip().replace("\n","")
    url = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"
    data = {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "grant_type": "refresh_token",
        "refresh_token": OFFLINE_TOKEN,
        "scope": "offline_access openid"
    }

    response = requests.post(url, data=data)
    response.raise_for_status()

    response_json = response.json()
    access_token = response_json.get('access_token')
    print("Access token retrieved successfully.")

    if not access_token:
        raise RuntimeError("Failed to retrieve access token from IAM response")

else:
    raise RuntimeError(f"No token file found in {os.getcwd()} !")

Select which products to explore:
- From the products_rfi.json which are converted to a dataframe. As an example we select the unmitigated proucts with an iso_impact larger than 10%
- Select any of the hard-coded list
- Conventional filtering by bbox, dates etc.

In [ ]:
from processing import pol_mean
print(len(gdf))
print(len(products_rfi))
rows = [{"product_id": p["id"], "rfiCorrectionFlag": p["rfiCorrectionFlag"], **p["properties"], **p["fm"]}
        for p in products_rfi]
print(len(rows))
df_rfi = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["product_id"])
print(len(df_rfi))
for kind in ("pers", "iso"):
    df_rfi[f"{kind}_avgBW"] = pol_mean(df_rfi, kind, "avgBW")
    df_rfi[f"{kind}_linesPct"] = pol_mean(df_rfi, kind, "linesPct")
    df_rfi[f"{kind}_impact"] = df_rfi[f"{kind}_avgBW"] * df_rfi[f"{kind}_linesPct"] / 100
    df_rfi[f"{kind}_impact"] = df_rfi[f"{kind}_avgBW"] * df_rfi[f"{kind}_linesPct"] / 100
    
df_selected = df_rfi[df_rfi["rfiCorrectionFlag"] == "false"]

PRCT_IMPACT = 10.0
n_before = len(df_selected)
df_selected = df_selected[df_selected["iso_impact"] >= PRCT_IMPACT]
print(f"discarded {n_before - len(df_selected)} of {n_before} products "
      f"(iso_impact < {PRCT_IMPACT} or missing), {len(df_selected)} kept")

In [ ]:
# Middle east
middle_east = ['BIO_S1_SCS__1S_20260513T150807_20260513T150827_T_G01_M03_C07_T039_F134_02_DRE0O0', 
               'BIO_S1_SCS__1S_20260513T150904_20260513T150924_T_G01_M03_C07_T039_F137_02_DRDZV9', 
               'BIO_S1_SCS__1S_20260513T150826_20260513T150846_T_G01_M03_C07_T039_F135_02_DRE0HI', 
               'BIO_S3_SCS__1S_20260829T150437_20260829T150457_T_G01_M04_C05_T039_F107_02_DWY0RR',
               'BIO_S3_SCS__1S_20260829T150359_20260829T150419_T_G01_M04_C05_T039_F105_02_DWY14T']
# Ecuador
ecuador = [
    "BIO_S3_SCS__1S_20260829T233255_20260829T233313_T_G01_M04_C05_T044_F162_02_DWYO99",
    "BIO_S3_SCS__1S_20260829T233236_20260829T233257_T_G01_M04_C05_T044_F161_02_DWYO8H",
    "BIO_S3_SCS__1S_20260829T233217_20260829T233238_T_G01_M04_C05_T044_F160_02_DWYO38",
    "BIO_S3_SCS__1S_20260829T233158_20260829T233219_T_G01_M04_C05_T044_F159_02_DWYO4X",
    "BIO_S3_SCS__1S_20260829T233139_20260829T233200_T_G01_M04_C05_T044_F158_02_DWYO5W",
    "BIO_S3_SCS__1S_20260829T233120_20260829T233141_T_G01_M04_C05_T044_F157_02_DWYO5M",
    "BIO_S3_SCS__1S_20260829T233101_20260829T233122_T_G01_M04_C05_T044_F156_02_DWYO3C",
    "BIO_S3_SCS__1S_20260829T233042_20260829T233103_T_G01_M04_C05_T044_F155_02_DWYOAL",
    "BIO_S3_SCS__1S_20260829T233023_20260829T233044_T_G01_M04_C05_T044_F154_02_DWYO9B",
    "BIO_S3_SCS__1S_20260829T233004_20260829T233025_T_G01_M04_C05_T044_F153_02_DWYOAY",
]

selected = [
    # Set 1, Region 1
    "BIO_S1_SCS__1S_20260915T104023_20260915T104044_T_G01_M05_C01_T021_F293_02_DXTLHJ",
    "BIO_S1_SCS__1S_20260918T104025_20260918T104046_T_G01_M05_C02_T021_F293_02_DXZAG1",
    "BIO_S1_SCS__1S_20260921T104028_20260921T104048_T_G01_M05_C03_T21_F293_02_DY4MWQ",
    "BIO_S1_SCS__1S_20260924T104030_20260924T104050_T_G01_M05_C04_T021_F293_02_DYA6ZC",
    "BIO_S1_SCS__1S_20260927T104032_20260927T104052_T_G01_M05_C05_T021_F293_02_DYFQMZ",
    "BIO_S1_SCS__1S_20260930T104034_20260930T104055_T_G01_M05_C06_T021_F293_02_DYLATM",
    "BIO_S1_SCS__1S_20260915T104101_20260915T104122_T_G01_M05_C01_T021_F295_02_DXTLHG",
    "BIO_S1_SCS__1S_20260918T104103_20260918T104124_T_G01_M05_C02_T021_F295_02_DXZAHN",
    "BIO_S1_SCS__1S_20260921T104106_20260921T104126_T_G01_M05_C03_T021_F295_02_DY4MZ4",
    "BIO_S1_SCS__1S_20260924T104108_20260924T104128_T_G01_M05_C04_T021_F295_02_DYA6ZD",
    "BIO_S1_SCS__1S_20260927T104110_20260927T104130_T_G01_M05_C05_T021_F295_02_DYFQOH",
    "BIO_S1_SCS__1S_20260930T104112_20260930T104133_T_G01_M05_C06_T021_F295_02_DYLAZI",
    # Set 1, Region 2
    "BIO_S1_SCS__1S_20260916T220549_20260916T220609_T_G01_M05_C01_T043_F184_02_DXW72Z",
    "BIO_S1_SCS__1S_20260919T220551_20260919T220611_T_G01_M05_C02_T043_F184_02_DY1SIO",
    "BIO_S1_SCS__1S_20260922T220553_20260922T220614_T_G01_M05_C03_T043_F184_02_DY79AQ",
    "BIO_S1_SCS__1S_20260925T220555_20260925T220616_T_G01_M05_C04_T043_F184_02_DYCSA3",
    "BIO_S1_SCS__1S_20260928T220558_20260928T220618_T_G01_M05_C05_T043_F184_02_DYICLW",
    # Set 2, Region 1
    "BIO_S1_SCS__1S_20260915T212528_20260915T212549_T_G01_M05_C01_T028_F160_02_DXUDDE",
    "BIO_S1_SCS__1S_20260918T212530_20260918T212551_T_G01_M05_C02_T028_F160_02_DY00H6",
    "BIO_S1_SCS__1S_20260921T212532_20260921T212553_T_G01_M05_C03_T028_F160_02_DY5EBB",
    "BIO_S1_SCS__1S_20260924T212535_20260924T212555_T_G01_M05_C04_T028_F160_02_DYAYBQ",
    "BIO_S1_SCS__1S_20260927T212537_20260927T212557_T_G01_M05_C05_T028_F160_02_DYGHV7",
    # Set 2, Region 2
    "BIO_S1_SCS__1S_20260915T090619_20260915T090640_T_G01_M05_C01_T020_F306_02_DXTDVD",
    "BIO_S1_SCS__1S_20260918T090622_20260918T090642_T_G01_M05_C02_T020_F306_02_DXZ169",
    "BIO_S1_SCS__1S_20260921T090624_20260921T090644_T_G01_M05_C03_T020_F306_02_DY4IY2",
    "BIO_S1_SCS__1S_20260924T090626_20260924T090646_T_G01_M05_C04_T020_F306_02_DYA1LO",
    "BIO_S1_SCS__1S_20260927T090628_20260927T090648_T_G01_M05_C05_T020_F306_02_DYFMCQ",
    "BIO_S1_SCS__1S_20260930T090630_20260930T090651_T_G01_M05_C06_T020_F306_02_DYL677",
]

bbox_south_america = [-81.5, -56.0, -34.0, 13.0]       # lists of floats, not strings
bbox_antarctica    = [-180, -90, 180, -60]

bbox = None # Set to valid bbox if filter with it
# Which products to load: "ids" (hard-coded list `selected`), "top_impact" (N largest iso_impact in products_rfi.json)
# or "threshold" (unmitigated products with iso_impact >= PRCT_IMPACT, from df_selected)
SELECT = "selected"
N = 5
if SELECT == "top_impact":
    ids = df_rfi.nlargest(N, "iso_impact")["product_id"].to_list()
elif SELECT == "random":
    ids = df_rfi.sample(N)["product_id"].to_list()
elif SELECT == "threshold":
    ids = df_selected["product_id"].to_list()
else:
    ids = selected
print(len(ids))
if bbox is not None:
    search = catalog.search(
        method="GET",
        collections=["BiomassLevel1a"],
        datetime=["2026-01-01T00:00:00Z", "2026-07-29T23:33:00Z"],
        bbox=bbox,
        max_items=len(ids),
    )
    items = list(search.items())
    print(f"{len(items)} items accessed (max_items=10), {search.matched()} matched the query.")
    data = search.item_collection_as_dict()
else:
    B = 100
    searches = [catalog.search(method="POST", ids=ids[i:i + B], limit=B) for i in range(0, len(ids), B)]
    items = [it for s in searches for it in s.items()]
    missing = set(ids) - {it.id for it in items}
    print(f"{len(items)} of {len(ids)} ids found, {len(missing)} missing.")
    # convert to df for easy visualisation
    data = {"type": "FeatureCollection",
        "features": [f for s in searches for f in s.items_as_dicts()]}

In [ ]:
df = pd.json_normalize(data, record_path=["features"])[
    [
        "id",  # unique id
        "properties.eopf:datatake_id",  # not unique id
        "properties.product:type",
        "properties.updated",
        "properties.sat:absolute_orbit",
        "properties.sat:orbit_state",
        # "assets.thumbnail.href",
        "assets.enclosure_kml.href",
        "assets.quicklook_jpg.href",
        "assets.enclosure_i_abs_tiff.href",
        "assets.enclosure_i_phase_tiff.href",
        "assets.product.href",
    ]
]

# Renaming the assets for
df.rename(
    columns={
        "properties.eopf:datatake_id": "dt_id",
        "properties.product:type": "product_type",
        "properties.updated": "last_modified",
        "properties.sat:absolute_orbit": "abs_orbit",
        "properties.sat:orbit_state": "orbit_state",
        # "assets.thumbnail.href": "thumbnail",
        "assets.enclosure_kml.href": "quicklook_kml",
        "assets.quicklook_jpg.href": "quicklook_jpg",
        "assets.enclosure_i_abs_tiff.href": "abs_product",
        "assets.enclosure_i_phase_tiff.href": "phase_product",
        "assets.product.href": "zipped_product",
    },
    inplace=True,
)

df.sort_values(by="id", ascending=True, ignore_index=True, inplace=True)
df

In [ ]:
by_id = {it.id: it for it in items}
items_selected = [by_id[i] for i in ids if i in by_id]
print(f"{len(items_selected)} products to process")

In [ ]:
N = 10
items_selected = list(items)[:N]          # the first N
print(f"{len(items_selected)} products to process")

## Load the product

In [ ]:
from pathlib import Path

KEYS = ["enclosure_annot_xml", "enclosure_i_abs_tiff", "enclosure_i_phase_tiff", "quicklook_jpg"]

# Already downloaded: just resolve the local paths of each product
paths_by_id = {
    item.id: {k: data_path / item.id / Path(item.assets[k].href).name for k in KEYS}
    for item in items_selected
}

missing = [p for d in paths_by_id.values() for p in d.values() if not p.exists()]
print(len(paths_by_id), "products,", len(missing), "missing files")

In [ ]:
# 2) read one product on demand (not all at once, a full SCS is large)
POLS = ["HH", "HV", "VH", "VV"]   # assumed band order, check src.descriptions or the annotation XML

for item in items_selected:
    if item.id not in paths_by_id:
        continue                                                   # failed download
    slc = read_item(item.id, paths_by_id)
    print(slc.shape)
    plot_product(slc, item.id, fetch_quicklook(item, paths_by_id[item.id]["quicklook_jpg"]), out_dir = f"{out_path}/{item.id}")
    del slc

The product must have been downloaded in the plot_rfi_product notebook

Detect with Lambda-1 Detector and Supress the RFI with BSF
source: <https://github.com/huizhangyang/lambda1-bsf/tree/main>

### Single run with default hyperparameters

In [ ]:
from lambda1_bsf.yang_filter import lamdba1_bsf

def filter_lambda1_bsf(slc, tile=8000, lblk=(500, 500), lpd=(50, 50), gamma=2.5,
                       ns_max=15, ns_mul=2, ups1=1.5, ups2=1.5, bdr1=0.7, bdr2=0.7):
    out = np.empty_like(slc)
    mask = np.zeros(slc.shape, dtype=np.int8)
    for y in range(0, slc.shape[0], tile):
        for x in range(0, slc.shape[1], tile):
            out[y:y+tile, x:x+tile], mask[y:y+tile, x:x+tile] = lamdba1_bsf(
                slc[y:y+tile, x:x+tile], list(lblk), list(lpd), gamma, ns_max, ns_mul,
                ups1, ups2, bdr1, bdr2)
    return out, mask

def ups_bdr(annot_xml):
    """ups/bdr for lambda1: axis 1 = azimuth, axis 2 = range."""
    root = ET.parse(annot_xml).getroot()

    def get(path):
        el = root.find(path)
        if el is None:
            raise KeyError(f"tag not found: {path}")
        return float(el.text)

    az, rg = "azimuthProcessingParameters", "rangeProcessingParameters"
    bdr = lambda blk: (lambda a: a**2 + (1 - a) ** 2 / 2)(get(f".//{blk}/windowCoefficient"))  # Hamming: B_eq/B
    return {
        "ups1": get(".//prf/value") / get(f".//{az}/processingBandwidth"),          # PRF / B_az
        "ups2": get(f".//{rg}/totalBandwidth") / get(f".//{rg}/processingBandwidth"),  # fs_rg / B_rg
        "bdr1": bdr(az),
        "bdr2": bdr(rg),
    }

In [ ]:
POLS = ["HH", "HV", "VH", "VV"]   # assumed band order, check src.descriptions or the annotation XML

selected_pol = "HV"

for item in tqdm(items_selected):
    if item.id not in paths_by_id:
        continue
    p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
    print(p)
    slc = read_item(item.id, paths_by_id)
    slc_clean, mask = filter_lambda1_bsf(slc[POLS.index(selected_pol)], **p)
    plot_result(slc[POLS.index(selected_pol)], 
                slc_clean, mask, 
                max_px = 2000, 
                aspect = "auto", 
                title=f"{item.id} pol {selected_pol}",
                out_dir = f"{out_path}/{item.id}_lambda1_bsf")
    del slc, slc_clean, mask

### Sweep over hyperparameters

In [ ]:
import pandas as pd
from itertools import product
from scipy.ndimage import median_filter

def ie(x, edges):                                  # intensity entropy, lower = better
    h, _ = np.histogram(x, bins=edges); p = h[h > 0] / h.sum()
    return -(p * np.log2(p)).sum()

def ic(x):                                         # coefficient of variation of intensity
    return x.std() / x.mean()

def blind_metrics(slc, clean, mask):
    m = mask.astype(bool)
    if not m.any():
        return {}
    I0, I1 = np.abs(slc[m]) ** 2, np.abs(clean[m]) ** 2          # flagged pixels only
    edges = np.linspace(0, np.percentile(I0, 99.5), 129)          # same bins before/after
    p_rfi = (np.abs(slc[m] - clean[m]) ** 2).sum()
    return {
        "mnr_dB": 10 * np.log10(p_rfi / I1.sum()),                # P_removed / P_clean, lower = better
        "IE_gain": ie(I0, edges) - ie(I1, edges),                 # >0 = entropy dropped
        "IC_ratio": ic(I1) / ic(I0),
    }
    
def row_rough(slc, k=101):
    """Streak metric: std of the row-mean dB profile minus its running median. Lower = fewer streaks."""
    p = 10 * np.log10((np.abs(slc) ** 2).mean(axis=1) + 1e-12)   # axis=1 -> one value per azimuth row
    return np.std(p - median_filter(p, k, mode="nearest"))

def metrics(slc, clean, mask, rough_src):
    p_src = (np.abs(slc) ** 2).sum()
    return {
        "flag_frac": mask.mean(),                                # fraction of pixels in flagged blocks
        "removed_frac": (np.abs(slc - clean) ** 2).sum() / p_src, # energy removed / total energy
        "rough_ratio": row_rough(clean) / rough_src,             # <1: streaks reduced
    }

grid = {
    "lblk":   [(500, 500)],
    "lpd":    [(50, 50)],
    "gamma":  [2.5, 5, 10, 20],
    "ns_mul": [1, 2],
}
show = {(2.5, 2), (20, 2)}          # (gamma, ns_mul) runs to inspect visually

rows = []
for item in tqdm(items_selected):
    if item.id not in paths_by_id:
        continue
    slc = read_item(item.id, paths_by_id)[POLS.index(selected_pol)]
    rough_src = row_rough(slc)
    for values in product(*grid.values()):
        params = dict(zip(grid, values))
        p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
        print(p)
        clean, mask = filter_lambda1_bsf(slc, **params, **p)
        rows.append({"id": item.id, **params, **metrics(slc, clean, mask, rough_src),
                     **blind_metrics(slc, clean, mask)})
        if (params["gamma"], params["ns_mul"]) in show:
            plot_result(slc, clean, mask, aspect="auto",
                        title=f"{item.id} gamma={params['gamma']} ns_mul={params['ns_mul']}")
        del clean, mask
    del slc

df = pd.DataFrame(rows)

In [ ]:
df["cfg"] = "ns_mul" + df.ns_mul.astype(str)
cols = [c for c in ["flag_frac", "removed_frac", "rough_ratio", "mnr_dB", "IE_gain", "IC_ratio"] if c in df]

for pid, d in df.groupby("id"):
    fig, ax = plt.subplots(1, len(cols) + 1, figsize=(4 * (len(cols) + 1), 3.5), constrained_layout=True)
    for cfg, g in d.groupby("cfg"):
        g = g.sort_values("gamma")
        for a, c in zip(ax, cols):                       # one metric vs gamma, line per config
            a.plot(g.gamma, g[c], "o-", label=cfg)
        ax[-1].scatter(g.removed_frac, g.rough_ratio, label=cfg)   # trade-off: cost vs benefit
        for _, r in g.iterrows():
            ax[-1].annotate(f"γ{r.gamma}", (r.removed_frac, r.rough_ratio), fontsize=7)
    for a, c in zip(ax, cols):
        a.set(title=c, xlabel="gamma")
    ax[-1].set(title="trade-off (lower-left = better)", xlabel="removed_frac", ylabel="rough_ratio")
    ax[0].legend(fontsize=7)
    fig.suptitle(pid)
    plt.show()

In [ ]:
def block_margins(slc, lblk=(500, 500), lpd=(50, 50), **kw):
    """Per-block lambda1 / threshold (gamma=1). A block is flagged at `gamma` if margin > gamma."""
    R, C = slc.shape
    out = []
    for y in range(0, R, lblk[0]):
        for x in range(0, C, lblk[1]):
            P = slc[max(y - lpd[0], 0):min(y + lblk[0] + lpd[0], R),
                    max(x - lpd[1], 0):min(x + lblk[1] + lpd[1], C)].copy()
            if min(P.shape) < 100:                              # skip tiny edge slivers
                continue
            a = np.abs(P); P[a > a.mean() + 8 * a.std()] = 0    # same outlier rule as the repo
            out.append((y, x, lam1(P) / get_eig_th(np.var(P), *P.shape, gamma=1, **kw)))
    return np.array(out)

def patch(ch, y, x, lblk=(500, 500), lpd=(50, 50)):
    R, C = ch.shape
    P = ch[max(y - lpd[0], 0):min(y + lblk[0] + lpd[0], R),
           max(x - lpd[1], 0):min(x + lblk[1] + lpd[1], C)].copy()
    a = np.abs(P); P[a > a.mean() + 8 * a.std()] = 0                  # same outlier rule as the repo
    return P

def plot_spectrum(ch, blocks, gammas=(1, 2.5, 10), top=100, **kw):
    """blocks: list of (y, x, label). Eigenvalues lambda = s^2 of each block vs the thresholds."""
    fig, ax = plt.subplots(1, len(blocks), figsize=(5 * len(blocks), 4), sharey=True, constrained_layout=True)
    for a, (y, x, label) in zip(np.atleast_1d(ax), blocks):
        P = patch(ch, y, x)
        lam = np.linalg.svd(P, compute_uv=False) ** 2
        n, q = P.shape; var = np.var(P)
        a.semilogy(np.arange(1, top + 1), lam[:top], "o", ms=3, color="C0")
        for g, c in zip(gammas, ("C1", "C2", "C3")):
            th = get_eig_th(var, n, q, gamma=g, **kw)
            a.axhline(th, ls="--", color=c, label=f"γ={g}: th, q={int((lam > th).sum())}")
        a.axhline((np.sqrt(n) + np.sqrt(q)) ** 2 * var, ls=":", color="gray", label="white-noise MP edge")
        a.set(title=f"{label}  (y={y}, x={x})", xlabel="eigenvalue index")
        a.legend(fontsize=7)
    np.atleast_1d(ax)[0].set_ylabel("λ = s²")
    plt.show()

for item in tqdm(items_selected):
    if item.id not in paths_by_id:
        continue
    p = ups_bdr(paths_by_id[item.id]["enclosure_annot_xml"])
    print(p)
    ch = read_item(item.id, paths_by_id)[POLS.index(selected_pol)]
    mm = block_margins(ch, **p)
    m_real = mm[:, 2]
    
    w = (p["bdr1"] / p["ups1"], p["bdr2"] / p["ups2"])             # occupied band fraction per axis
    m_sim = np.array([lam1(X) / get_eig_th(np.var(X), *X.shape, gamma=1, **p)
                      for X in (cnoise(600, 600, w) for _ in range(100))])
    
    bins = np.linspace(np.log10(min(m_real.min(), m_sim.min())), np.log10(m_real.max()), 40)
    plt.hist(np.log10(m_sim), bins, alpha=0.6, label="simulated H0 noise", density=True)
    plt.hist(np.log10(m_real), bins, alpha=0.6, label=f"real blocks (n={len(m_real)})", density=True)
    for g in (1, 2.5, 10):
        plt.axvline(np.log10(g), color="k", ls="--", lw=0.8); plt.text(np.log10(g), plt.ylim()[1] * 0.9, f"γ={g}", fontsize=8)
    plt.xlabel("log10(λ1 / threshold at γ=1)"); plt.legend(); plt.show()
    print({g: round((m_real > g).mean(), 3) for g in (1, 2.5, 5, 10, 20)})   # blocks flagged per gamma
    print("sim max margin:", m_sim.max())
    
    o = np.argsort(mm[:, 2])
    pick = [(int(mm[i, 0]), int(mm[i, 1]), lab) for i, lab in
            [(o[0], "lowest margin"), (o[len(o) // 2], "median"), (o[-1], "highest margin")]]
    plot_spectrum(ch, pick, **p)